# IPL 2008-2026: What Really Decides a Match?

## Questions
1. Does winning the toss help, and does it vary by venue or bat/field decision?
2. How has scoring changed across seasons?
3. Powerplay vs middle vs death overs: where are matches decided?
4. Which venues favour batters vs bowlers?
5. Is chasing more successful now than in 2008?
6. Who are the most impactful batters and bowlers?

# Data Cleaning

## import the required libraries

In [1]:
import pandas as pd
import os

## Load the data

In [2]:
balls        = pd.read_csv('../data/ball_by_ball_data.csv')
matches      = pd.read_csv('../data/ipl_matches_data.csv')
players      = pd.read_csv('../data/players-data-updated.csv')
team_aliases = pd.read_csv('../data/team_aliases.csv')
teams_data   = pd.read_csv('../data/teams_data.csv')

## View the Data

In [3]:
balls.sample(5)

,season_id,match_id,batter,bowler,non_striker,team_batting,team_bowling,over_number,ball_number,batter_runs,...,is_bye,is_penalty,wide_ball_runs,no_ball_runs,leg_bye_runs,bye_runs,penalty_runs,wicket_kind,is_super_over,innings
120264,2024,1426291,Shashank Singh,MJ Santner,P Simran Singh,494,129,7,1,1,...,False,False,0,0,0,0,0,NaN,False,2
197048,2012,548351,SR Tendulkar,A Nehra,RJ Peterson,3,1419,11,5,0,...,False,False,0,0,0,0,0,caught,False,1
84819,2022,1304101,RD Gaikwad,SN Thakur,DP Conway,129,252,5,4,4,...,False,False,0,0,0,0,0,NaN,False,1
113678,2024,1426264,A Badoni,Mukesh Kumar,Arshad Khan,614,252,18,1,4,...,False,False,0,0,0,0,0,NaN,False,1
258665,2016,980967,S Dhawan,DJ Bravo,NV Ojha,2,5,17,4,1,...,False,False,0,0,0,0,0,NaN,False,2


In [4]:
matches.sample(5)

,match_id,season_id,balls_per_over,city,match_date,event_name,match_number,gender,match_type,format,...,venue,toss_winner,team1,team2,toss_decision,match_winner,win_by_runs,win_by_wickets,result,player_of_match
1164,1473508,2025,6,New Chandigarh,2025-05-29,Indian Premier League,NaN,male,T20,T20,...,Maharaja Yadavindra Singh International Cricke...,1,494,1,field,1,NaN,8.0,win,8475.0
721,501219,2011,6,Kolkata,2011-04-20,Indian Premier League,22.0,male,T20,T20,...,Eden Gardens,6,6,1414,field,1414,6.0,NaN,win,12032.0
644,419109,2010,6,Kolkata,2010-03-14,Indian Premier League,4.0,male,T20,T20,...,Eden Gardens,6,6,1,field,6,NaN,7.0,win,39.0
1029,829797,2015,6,Chennai,2015-05-10,Indian Premier League,47.0,male,T20,T20,...,"MA Chidambaram Stadium, Chepauk",129,129,134,bat,129,12.0,NaN,win,348.0
780,548312,2012,6,Jaipur,2012-04-08,Indian Premier League,7.0,male,T20,T20,...,Sawai Mansingh Stadium,6,134,6,field,134,22.0,NaN,win,12318.0


In [5]:
players.sample(5)

,player_id,player_name,bat_style,bowl_style,field_pos,player_full_name
8,39,MK Tiwary,Right hand Bat,Legbreak Googly,,Manoj Kumar Tiwary
415,12669,LA Pomersbach,Left hand Bat,Right arm Offbreak,,Luke Anthony Pomersbach
157,8389,Arshad Khan,Left hand Bat,Left arm Medium,NaN,Arshad Khan
132,10065,R Ravindra,Left hand Bat,Slow Left arm Orthodox,NaN,Rachin Ravindra
653,3793,J Suchith,Left hand Bat,Slow Left arm Orthodox,,Jagadeesha Suchith


In [6]:
team_aliases.sample(5)

,alias_id,team_id,alias_name
24,25,4,Pune
23,24,4,RPS
9,10,252,DD
31,32,129,CSK
4,5,1,Royal Challengers Bangalore


In [7]:
teams_data

,team_id,team_name
0,1,Royal Challengers Bangalore
1,2,Sunrisers Hyderabad
2,3,Mumbai Indians
3,4,Rising Pune Supergiant
4,5,Gujarat Lions
5,6,Kolkata Knight Riders
6,129,Chennai Super Kings
7,134,Rajasthan Royals
8,252,Delhi Capitals
9,494,Punjab Kings


## View Basic info about the Data

In [8]:
dfs = {
    "balls"        : balls, 
    "matches"      : matches, 
    "players"      : players, 
    "team_aliases" : team_aliases, 
    "teams_data"   : teams_data
}

In [9]:
# Shape & columns name
for df_name, df in dfs.items():
    print('='*130)
    print('Data Frame name:',df_name)
    print('Shape of Data:',df.shape)
    print('\nColumns names:', df.columns.to_list())

Data Frame name: balls
Shape of Data: (288226, 30)

Columns names: ['season_id', 'match_id', 'batter', 'bowler', 'non_striker', 'team_batting', 'team_bowling', 'over_number', 'ball_number', 'batter_runs', 'extras', 'total_runs', 'batsman_type', 'bowler_type', 'player_out', 'fielders_involved', 'is_wicket', 'is_wide_ball', 'is_no_ball', 'is_leg_bye', 'is_bye', 'is_penalty', 'wide_ball_runs', 'no_ball_runs', 'leg_bye_runs', 'bye_runs', 'penalty_runs', 'wicket_kind', 'is_super_over', 'innings']
Data Frame name: matches
Shape of Data: (1212, 23)

Columns names: ['match_id', 'season_id', 'balls_per_over', 'city', 'match_date', 'event_name', 'match_number', 'gender', 'match_type', 'format', 'overs', 'season', 'team_type', 'venue', 'toss_winner', 'team1', 'team2', 'toss_decision', 'match_winner', 'win_by_runs', 'win_by_wickets', 'result', 'player_of_match']
Data Frame name: players
Shape of Data: (799, 6)

Columns names: ['player_id', 'player_name', 'bat_style', 'bowl_style', 'field_pos', 'pl

In [10]:
# info
for df_name, df in dfs.items():
    print('='*130)
    print("\nData Frame name:",df_name)
    print(df.info())


Data Frame name: balls
<class 'pandas.DataFrame'>
RangeIndex: 288226 entries, 0 to 288225
Data columns (total 30 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   season_id          288226 non-null  int64
 1   match_id           288226 non-null  int64
 2   batter             288226 non-null  str  
 3   bowler             288226 non-null  str  
 4   non_striker        288226 non-null  str  
 5   team_batting       288226 non-null  int64
 6   team_bowling       288226 non-null  int64
 7   over_number        288226 non-null  int64
 8   ball_number        288226 non-null  int64
 9   batter_runs        288226 non-null  int64
 10  extras             288226 non-null  int64
 11  total_runs         288226 non-null  int64
 12  batsman_type       287987 non-null  str  
 13  bowler_type        287887 non-null  str  
 14  player_out         14340 non-null   str  
 15  fielders_involved  14340 non-null   str  
 16  is_wicket          288226

In [11]:
# missing & Duplicate Values
for df_name, df in dfs.items():
    print('='*130)
    print("Data Frame name:",df_name)

    missing_cols = df.isna().sum()
    total_missing_values = missing_cols.sum()

    if total_missing_values > 0:
        print("\nMissing Values of Each Columns:")
        print(missing_cols[missing_cols > 0])
        print("\nTotal Missing values:",total_missing_values)
    else:
        print("NO Missing values")

    print("\nTotal Duplicate values:",df.duplicated().sum())
    

Data Frame name: balls

Missing Values of Each Columns:
batsman_type            239
bowler_type             339
player_out           273886
fielders_involved    273886
wicket_kind          273886
dtype: int64

Total Missing values: 822236

Total Duplicate values: 0
Data Frame name: matches

Missing Values of Each Columns:
city                51
match_number        70
win_by_runs        666
win_by_wickets     571
player_of_match      9
dtype: int64

Total Missing values: 1367

Total Duplicate values: 0
Data Frame name: players

Missing Values of Each Columns:
bowl_style    19
field_pos     88
dtype: int64

Total Missing values: 107

Total Duplicate values: 0
Data Frame name: team_aliases
NO Missing values

Total Duplicate values: 0
Data Frame name: teams_data
NO Missing values

Total Duplicate values: 0


## Statistical info

In [12]:
matches.describe().T

,count,mean,std,min,25%,50%,75%,max
match_id,1212.0,961685.974422,390532.810455,335982.0,548362.75,1082620.5,1304079.25,1529286.0
season_id,1212.0,2017.018152,5.439222,2008.0,2012.00,2017.0,2022.00,2026.0
balls_per_over,1212.0,6.000000,0.000000,6.0,6.00,6.0,6.00,6.0
match_number,1142.0,31.328371,18.397497,1.0,16.00,31.0,46.00,72.0
overs,1212.0,20.000000,0.000000,20.0,20.00,20.0,20.00,20.0
toss_winner,1212.0,175.825083,267.640775,1.0,3.00,129.0,252.00,1419.0
team1,1212.0,180.485149,275.288484,1.0,3.00,129.0,252.00,1419.0
team2,1212.0,184.131188,275.467446,1.0,3.00,129.0,252.00,1419.0
match_winner,1212.0,170.292079,250.118969,1.0,3.00,129.0,252.00,1419.0
win_by_runs,546.0,30.608059,27.010366,1.0,11.00,22.0,42.00,146.0


In [13]:
balls.describe().T

,count,mean,std,min,25%,50%,75%,max
season_id,288226.0,2017.033165,5.430094,2008.0,2012.0,2017.0,2022.0,2026.0
match_id,288226.0,962652.661228,390101.815920,335982.0,548364.0,1082623.0,1304080.0,1529286.0
team_batting,288226.0,182.616017,274.658815,1.0,3.0,129.0,252.0,1419.0
team_bowling,288226.0,181.611912,274.556424,1.0,3.0,129.0,252.0,1419.0
over_number,288226.0,9.190375,5.680521,0.0,4.0,9.0,14.0,19.0
ball_number,288226.0,2.628160,1.818354,0.0,1.0,3.0,4.0,10.0
batter_runs,288226.0,1.284638,1.658509,0.0,0.0,1.0,1.0,6.0
extras,288226.0,0.068217,0.342774,0.0,0.0,0.0,0.0,7.0
total_runs,288226.0,1.352855,1.644513,0.0,0.0,1.0,1.0,7.0
wide_ball_runs,288226.0,0.039531,0.255583,0.0,0.0,0.0,0.0,5.0


In [14]:
matches.describe(include="object").T

C:\Users\jaggi\AppData\Local\Temp\ipykernel_2524\2660587672.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  matches.describe(include="object").T


,count,unique,top,freq
city,1161,37,Mumbai,185
match_date,1212,920,2017-04-08,2
event_name,1212,1,Indian Premier League,1212
gender,1212,1,male,1212
match_type,1212,1,T20,1212
format,1212,1,T20,1212
season,1212,19,2013,76
team_type,1212,1,club,1212
venue,1212,59,Eden Gardens,77
toss_decision,1212,2,field,800


In [15]:
balls.describe(include="object").T

C:\Users\jaggi\AppData\Local\Temp\ipykernel_2524\1284082023.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  balls.describe(include="object").T


,count,unique,top,freq
batter,288226,727,V Kohli,6942
bowler,288226,568,R Ashwin,4868
non_striker,288226,715,V Kohli,6926
batsman_type,287987,2,Right hand Bat,189894
bowler_type,287887,21,Right arm Medium,49776
player_out,14340,676,RG Sharma,241
fielders_involved,14340,1129,[None],3908
wicket_kind,14340,10,caught,9056


In [16]:
# rename the old team name with new name
SHORT = {
    "Chennai Super Kings": "CSK",
    "Mumbai Indians": "MI",
    "Royal Challengers Bangalore": "RCB",
    "Kolkata Knight Riders": "KKR",
    "Delhi Capitals": "DC",
    "Punjab Kings": "PBKS",
    "Rajasthan Royals": "RR",
    "Sunrisers Hyderabad": "SRH",
    "Gujarat Titans": "GT",
    "Lucknow Super Giants": "LSG",
    "Deccan Chargers": "DCH",
    "Kochi Tuskers Kerala": "KTK",
    "Pune Warriors": "PWI",
    "Rising Pune Supergiant": "RPS",
    "Gujarat Lions": "GL",
}

team_names = team_aliases.merge(teams_data, on='team_id', how='left')
team_names = (team_names[['team_id', 'team_name']]
              .drop_duplicates(subset='team_name')
              .reset_index(drop=True))

team_names['short_name'] = team_names['team_name'].map(SHORT)

team_names

,team_id,team_name,short_name
0,1,Royal Challengers Bangalore,RCB
1,252,Delhi Capitals,DC
2,494,Punjab Kings,PBKS
3,2,Sunrisers Hyderabad,SRH
4,3,Mumbai Indians,MI
5,4,Rising Pune Supergiant,RPS
6,5,Gujarat Lions,GL
7,6,Kolkata Knight Riders,KKR
8,129,Chennai Super Kings,CSK
9,134,Rajasthan Royals,RR


# Cleaning

In [17]:
# copy of data before cleaning: so we can later reuse the original data
balls_c   = balls.copy()
matches_c = matches.copy()
players_c = players.copy()

dfs = {
    'balls_c'  : balls_c,
    'matches_c': matches_c,
    'players_c': players_c,
}

In [18]:
# checking the Datatypes
matches_c['match_date'] = pd.to_datetime(matches_c['match_date'])

In [19]:
# handling missing values
for df_name, df in dfs.items():
    print('='*100)
    print('DF Name:',df_name)
    cols = df.isna().sum()
    print(cols[cols > 0])

DF Name: balls_c
batsman_type            239
bowler_type             339
player_out           273886
fielders_involved    273886
wicket_kind          273886
dtype: int64
DF Name: matches_c
city                51
match_number        70
win_by_runs        666
win_by_wickets     571
player_of_match      9
dtype: int64
DF Name: players_c
bowl_style    19
field_pos     88
dtype: int64


In [20]:
# Filling missing values
balls_c['batsman_type'] = balls_c['batsman_type'].fillna('NA')
balls_c['bowler_type'] = balls_c['bowler_type'].fillna('NA')
balls_c['player_out'] = balls_c['player_out'].fillna('NA')
balls_c['fielders_involved'] = balls_c['fielders_involved'].fillna('NA')
balls_c['wicket_kind'] = balls_c['wicket_kind'].fillna('NA')
balls_c['is_super_over'] = balls_c['is_super_over'].replace({True:1, False:0}).astype('int32')
balls_c['is_wicket'] = balls_c['is_wicket'].replace({True:1, False:0}).astype('int32')
balls_c['is_wide_ball'] = balls_c['is_wide_ball'].replace({True:1, False:0}).astype('int32')
balls_c['is_no_ball'] = balls_c['is_no_ball'].replace({True:1, False:0}).astype('int32')
balls_c['is_leg_bye'] = balls_c['is_leg_bye'].replace({True:1, False:0}).astype('int32')
balls_c['is_bye'] = balls_c['is_bye'].replace({True:1, False:0}).astype('int32')
balls_c['is_penalty'] = balls_c['is_penalty'].replace({True:1, False:0}).astype('int32')

In [21]:
# where the next missing value is prev_value + 1
matches_c['year'] = matches_c['match_date'].dt.year

def fill_sequence(s):
    prev = s.ffill()                                   # last known number
    gap = s.isna().groupby(s.notna().cumsum()).cumsum()  # 1, 2, 3 within each run of NaNs
    return prev + gap

matches_c['match_number'] = (matches_c.groupby('year')['match_number']
                             .transform(fill_sequence)
                             .fillna(0)                # only if a season starts with a gap
                             .astype(int))


matches_c['win_by_runs'] = matches_c['win_by_runs'].fillna(0).astype(int)
matches_c['win_by_wickets'] = matches_c['win_by_wickets'].fillna(0).astype(int)
matches_c['player_of_match'] = matches_c['player_of_match'].fillna(-1).astype(int) # Beacuse match had no result

'''
# AS we can see that only for 2 venue there missing city.
# So, for venue: Dubai International Cricket Stadium the city is Dubai as per research.
# Also, for venue: Sharjah Cricket Stadium the city is Sharjah as per research.
Code: matches_c[matches_c['city'].isna()][['city','venue']]
'''
city_fix = {
    'Dubai International Cricket Stadium': 'Dubai',
    'Sharjah Cricket Stadium': 'Sharjah',
}

mask = matches_c['city'].isna()
matches_c.loc[mask, 'city'] = matches_c.loc[mask, 'venue'].map(city_fix)

In [22]:
players_c['field_pos'] = players_c['field_pos'].fillna('NULL')
players_c['bowl_style'] = players_c['bowl_style'].fillna('NULL')

In [23]:
# handling duplicate values
for df_name, df in dfs.items():
    print('='*100)
    print('DF name:',df_name, ", ", "Total Duplicate values:",df.duplicated().sum())

DF name: balls_c ,  Total Duplicate values: 0
DF name: matches_c ,  Total Duplicate values: 0
DF name: players_c ,  Total Duplicate values: 0


In [24]:
# No Duplicate values in all dataframe

In [25]:
# Drop the columns where are not usefull.
drop_column_balls   = ['wide_ball_runs', 'no_ball_runs','leg_bye_runs', 'bye_runs', 'penalty_runs', 'is_penalty']
drop_column_matches = ['balls_per_over', 'event_name', 'gender', 'match_type', 'format', 'overs', 'team_type'  ]
drop_column_players = ['player_full_name']

balls_c   = balls_c.drop(columns = drop_column_balls)
matches_c = matches_c.drop(columns = drop_column_matches)
players_c = players_c.drop(columns = drop_column_players)

In [26]:
# Rechecking Missing values & Duplicate values
for df_name, df in dfs.items():
    print('='*100)
    print('DF Name:',df_name)
    print("Total Duplicate values:",df.duplicated().sum())
    
    cols = df.isna().sum()
    print(cols[cols > 0])

DF Name: balls_c
Total Duplicate values: 0
Series([], dtype: int64)
DF Name: matches_c
Total Duplicate values: 0
Series([], dtype: int64)
DF Name: players_c
Total Duplicate values: 0
Series([], dtype: int64)


# Save the Cleaned Data

In [27]:
os.makedirs('../data/cleaned_data', exist_ok=True)

team_names.to_csv('../data/cleaned_data/team_names.csv', index=False)

matches_c.to_csv('../data/cleaned_data/matches_clean.csv', index=False)
balls_c.to_csv('../data/cleaned_data/ball_by_ball_clean.csv', index=False)
players_c.to_csv('../data/cleaned_data/players_clean.csv', index=False)

# See the total Shape & Memory useage, Before & After Cleaning the DF.

In [28]:
def mem_mb(d):
    return d.memory_usage(deep=True).sum() / (1024**2)

pairs = {
    'balls':   (balls,   balls_c),
    'matches': (matches, matches_c),
    'players': (players, players_c),
}

report = pd.DataFrame({
    'shape_before': {k: raw.shape for k, (raw, c) in pairs.items()},
    'shape_after':  {k: c.shape   for k, (raw, c) in pairs.items()},
    'memory_before_MB': {k: mem_mb(raw) for k, (raw, c) in pairs.items()},
    'memory_after_MB':  {k: mem_mb(c)   for k, (raw, c) in pairs.items()},
})
report

,shape_before,shape_after,memory_before_MB,memory_after_MB
balls,"(288226, 30)","(288226, 24)",69.521178,64.595573
matches,"(1212, 23)","(1212, 17)",0.319739,0.207754
players,"(799, 6)","(799, 5)",0.083851,0.063449


# Cleaning is finished